# Digital Advertising Performance & Client Insights
**Samir Sadman** — self-directed portfolio project  
**Stack:** Python/pandas • Excel • Tableau-ready CSV

> All campaign and financial data in this notebook is synthetic and created solely for portfolio demonstration.

## 1. Load and validate the campaign dataset
The dataset contains 756 daily campaign-market observations across six campaigns and three Canadian market groups.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("Samir_Tableau_Ready_Ad_Data.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

required = {"Date","Campaign","Objective","Market","Audience","Creative",
            "Spend","Impressions","Clicks","Conversions","Revenue","Video Views"}
assert required.issubset(df.columns)
assert not (df[["Spend","Impressions","Clicks","Conversions","Revenue"]] < 0).any().any()

df.head()

## 2. Aggregate campaign performance and calculate client KPIs
The core business metrics are CTR, CPM, CVR, CPA, ROAS and budget pacing.

In [ ]:
def add_kpis(frame):
    out = frame.copy()
    out["CTR"] = out["Clicks"] / out["Impressions"]
    out["CPM"] = out["Spend"] / out["Impressions"] * 1000
    out["CVR"] = out["Conversions"] / out["Clicks"]
    out["CPA"] = out["Spend"] / out["Conversions"]
    out["ROAS"] = out["Revenue"] / out["Spend"]
    return out

campaign = (df.groupby(["Campaign","Objective"], as_index=False)
              .agg({"Spend":"sum","Revenue":"sum","Impressions":"sum",
                    "Clicks":"sum","Conversions":"sum","Video Views":"sum"}))
campaign = add_kpis(campaign)

budgets = {
    "Creator Spark":45000, "Retargeting Boost":30000, "Product Demo":50000,
    "Broad Prospecting":55000, "UGC Story":40000, "Offer Test":35000
}
campaign["Budget"] = campaign["Campaign"].map(budgets)
campaign["Budget Pacing"] = campaign["Spend"] / campaign["Budget"]
campaign.sort_values("ROAS", ascending=False)

## 3. Executive findings
- **Retargeting Boost** leads efficiency at ~5.5x ROAS and ~$14.6 CPA.
- **Broad Prospecting** is the main efficiency drag at ~1.4x ROAS while spending above plan.
- **Creator Spark** is under-paced at ~83% of budget while producing ~4.5x ROAS, creating room to scale.
- The total portfolio produces ~3.4x blended ROAS with ~96.8% budget pacing.

In [ ]:
spend = campaign["Spend"].sum()
revenue = campaign["Revenue"].sum()
ctr = campaign["Clicks"].sum() / campaign["Impressions"].sum()
cpa = spend / campaign["Conversions"].sum()

print(f"Spend: ${spend:,.0f}")
print(f"Revenue: ${revenue:,.0f}")
print(f"ROAS: {revenue/spend:.2f}x")
print(f"CTR: {ctr:.2%}")
print(f"CPA: ${cpa:.2f}")

## 4. Client recommendation
Reallocate a controlled **10–15%** of Broad Prospecting spend toward Retargeting Boost and Creator Spark. Refresh Prospecting creative and landing-page alignment, then monitor marginal CPA before restoring spend. Keep Product Demo and UGC Story as stable coverage while scaling only after efficiency remains inside target.

## 5. Tableau handoff
`Samir_Tableau_Ready_Ad_Data.csv` is the clean row-level extract. Recommended Tableau sheets: **Portfolio KPI Strip**, **ROAS by Campaign**, **Spend vs Budget**, **CPA vs CTR**, **Weekly ROAS Trend**, and **Market Performance**.